In [13]:
!pip install -q sentence-transformers torch scikit-learn
import json, numpy as np, torch, torch.nn as nn
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings; warnings.filterwarnings("ignore")        # quiet sklearn/numpy chatter
np.seterr(all="ignore")
torch.manual_seed(0); np.random.seed(0)

# If running on Colab, clone the repo first:
# !([ -d bluedot-tais-puzzle ] || git clone https://github.com/SamDower/bluedot-tais-puzzle.git)
# %cd bluedot-tais-puzzle

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(384,64), nn.ReLU(),   # hidden 0
            nn.Linear(64,64),  nn.ReLU(),   # hidden 1
            nn.Linear(64,64),  nn.ReLU(),   # hidden 2  <- analyze post-ReLU here (layer L)
            nn.Linear(64,64),  nn.ReLU(),   # hidden 3
            nn.Linear(64,8),                # logits
        )
    def forward(self, x): return self.layers(x)

enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
m = Head(); m.load_state_dict(torch.load("model.pt", map_location="cpu", weights_only=False)); m.eval()
feature_names = json.load(open("feature_names.json"))

def load_jsonl(p):
    t, l = [], []
    for line in open(p):
        d = json.loads(line); t.append(d["text"]); l.append(d["labels"])
    return t, np.array(l)
tr_t, tr_y = load_jsonl("train.jsonl")
te_t, te_y = load_jsonl("test.jsonl")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10027.00it/s]


In [15]:
print(len(tr_t), "train /", len(te_t), "test examples; features:", feature_names)
print(tr_y.shape, te_y.shape)
print(*[f"{tr_y[:,i].sum()}/{len(tr_y[:,i])}" for i in range(8)])

7000 train / 1500 test examples; features: ['number', 'question', 'color', 'food', 'sentiment', 'country', 'person', 'body_part']
(7000, 8) (1500, 8)
3793/7000 3430/7000 3537/7000 3673/7000 3567/7000 3451/7000 3504/7000 3502/7000
